# Language Model and Sequence Generation — Transformers

In this lab, we study the different ways to decode text from a trained model. This step is needed for many natural language processing tasks: translation, question answering, generation…

We use text generation with GPT-2 as an example. GPT-2 is a *decoder-only* transformer: today's large language models (GPT-4, Llama, Mistral…) rely on the same principle and are decoded with the same methods.

We start from the simplest method and work up to the methods used in practice. This will give you a solid understanding of the role of the softmax at the output of language models.

An [excellent article](https://huggingface.co/blog/how-to-generate) by Hugging Face is a useful companion to this notebook.

**Warning**: these models were trained on unfiltered web text. Generated texts, especially with sampling, can be incoherent, false or offensive.

## Installation

The [`transformers`](https://github.com/huggingface/transformers) library by [Hugging Face](https://huggingface.co/) provides many models from the transformer family. It is preinstalled on Colab; the next cell just makes sure it is available.

In [ ]:
!pip install -q transformers

## Choice of language

We will generate text in English, but also in French, by loading models pretrained on different corpora. The next cell sets the language used in the rest of the lab.

In [ ]:
lang = "en"
# lang = "fr"

## Loading a trained model

Besides the source code of many transformer architectures, the `transformers` library lets you download [trained models](https://huggingface.co/models) from the Hugging Face Hub.

In this lab, we use the GPT-2 architecture (2019). The `from_pretrained` method, given the name of the model we want, downloads it. The tokenizer that goes with the model is loaded the same way.

PyTorch models are `nn.Module`s: we move them to the GPU with `.to(device)`, and switch them to evaluation mode with `.eval()` (which disables dropout).

In [ ]:
import functools

import torch
import tqdm.auto
import transformers

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Computing on: {device}")

pretraining_name = "antoiloui/belgpt2" if lang == "fr" else "gpt2"

model = transformers.AutoModelForCausalLM.from_pretrained(pretraining_name)
model = model.to(device).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(pretraining_name)

## Inspecting the model

After loading a pretrained model, it is always good practice to check its architecture and characteristics. We can print its configuration and the model itself (an `nn.Module` prints all its layers).

- *How many transformer layers does the model have?*
- *What are its regularization mechanisms?*
- *Its training stabilization mechanisms?*
- *How many parameters does it have?*
- *Later on, we need the index of the special token used during training to mark the beginning and end of a text. What is it?*

In [ ]:
print(model.config)
print(model)

*Your answer here.*
-
-
-
-
-

### Solution

In [ ]:
n_parameters = sum(p.numel() for p in model.parameters())
print(f"{n_parameters / 1e6:.0f} million parameters")
print(f"Layers: {model.config.n_layer}")
print(f"Beginning/end of text: {model.config.bos_token_id} / {model.config.eos_token_id}")

- It has 12 transformer layers (`n_layer`, or the 12 `GPT2Block`s printed).
- Its main regularization mechanism is dropout (`attn_pdrop`, `resid_pdrop`, `embd_pdrop`).
- Its training stabilization mechanisms are layer normalization (`LayerNorm`, `ln_1`, `ln_2`) and residual connections.
- It has about 124 million parameters.
- The indices of the beginning and end of text tokens are the `bos_token_id` and `eos_token_id` keys of the config.

## The generation mechanism

To generate text from a trained model, we proceed iteratively: we generate words one by one, starting from an empty text or from a sentence we want to complete.

To generate a word, we feed the network the words generated so far (as embedding indices). The network then outputs one score per vocabulary word. We pick one of these words, append it to what has already been generated, and start again to generate the next word.

More formally, we try to maximize $P(\text{Text}| \text{Initial})$ with the following decomposition:

$$P(\text{Text}| \text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Initial})$$

The whole point of decoding is to find the text that maximizes this product: it is the model's best proposal. This value can't be computed exactly, so we use heuristics to approximate it.

The transformer's **causal mask** is what makes this decomposition possible: during training, each position only sees the previous positions.

## Tokenization

The tokenizer loaded with the model turns a sentence into vocabulary indices (*subwords*) the model understands.

In the rest of the lab, we use the following `encode` and `decode` functions. `encode` adds the beginning of text token, and returns a tensor of shape `(1, number of tokens)` on the right device.

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  tokens = tokenizer.encode(sentence, add_special_tokens=False,
                            return_tensors="pt")
  bos = torch.tensor([[model.config.bos_token_id]])
  return torch.cat([bos, tokens], dim=1).to(device)


def decode(tokens: torch.Tensor) -> str:
  return tokenizer.decode(tokens.squeeze(0)[1:])


example = encode("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")
print(example)
print(tokenizer.convert_ids_to_tokens(example[0]))

## Studying the pretrained model's outputs

The first step to decode sentences from a trained model is to run a forward pass to get its predictions for the next word. Let's study how it behaves.

Like any PyTorch module, the forward pass is computed with `model(input_ids)`. The result is an object whose main attributes are:

- `logits`: the model's scores, before softmax;
- `past_key_values`: a cache of the attention keys and values already computed.

*Study the model's outputs ([documentation](https://huggingface.co/docs/transformers/model_doc/gpt2#transformers.GPT2LMHeadModel)) on various sequences, for instance:*

- *"" (the empty text)*
- *"I think, therefore I am" or "Je pense, donc je suis" depending on the model's language*

*What is the shape of the `logits`? What does each dimension correspond to?*

*Remember to disable gradient computation with `torch.no_grad()`: we are not training the model.*

In [ ]:
# Your code here

### Solution

In [ ]:
def describe_shapes(input_string: str) -> None:
  with torch.no_grad():
    output = model(encode(input_string))
  print("-" * 80)
  print(f"Logits shape for input '{input_string}': "
        f"{tuple(output.logits.shape)}")
  print(f"Cache type: {type(output.past_key_values).__name__}")


describe_shapes("")
describe_shapes("Je pense, donc je suis" if lang == "fr" else "I think, therefore I am")

The logits have shape `(batch, number of tokens, vocabulary size)`: for each input position, the model scores every vocabulary word for the **next** position. To generate, only the last position matters.

## A `forward` function suited to decoding

We use the following `forward` function, which returns a result suited to decoding: only the last position matters (the prediction of the next word).

It uses the `past_key_values` cache: it avoids recomputing, for every new word, the attention keys and values of all previous words. When the cache is given, **only the new tokens** must be passed to the model.

In [ ]:
@torch.no_grad()
def forward(tokens: torch.Tensor, past=None) -> tuple[torch.Tensor, object]:
  output = model(tokens, past_key_values=past, use_cache=True)
  return output.logits[:, -1, :], output.past_key_values

So use either:

- `forward(all_tokens)`
- `forward(last_token, past)`

where `all_tokens` would for instance be a sequence of 7 tokens of shape `(1, 7)` while `last_token` would have shape `(1, 1)`.

*Why is it often useful to keep dimensions of size `1` in neural networks?*

*Your answer here.*

### Solution

These dimensions make it possible to compute results for several examples in parallel (batching): the same code works for a batch of 1 as well as 64.

## Decoding loop

A decoding loop always follows the same pattern:

1. Encode everything produced so far (inputs and previous outputs)
2. Choose the index of the next word
3. Repeat 1. and 2. until a stopping criterion is met (in this lab, only the number of words produced)

Step 2 is where everything happens. The following code implements everything else, so that we can focus on step 2: each decoding method will be a function that receives the logits of the last word (shape `(1, vocabulary size)`) and returns the index of the chosen word (a tensor of shape `()`).

In [ ]:
def decoding_loop(step_function):

  @functools.wraps(step_function)
  def wrapper(prompt: str, length: int, *step_args, **step_kwargs) -> str:
    decoded = [encode(prompt)]
    past = None
    for _ in tqdm.auto.trange(length, desc="Words", leave=False):
      logits, past = forward(decoded[-1], past)
      index = step_function(logits, *step_args, **step_kwargs)
      decoded.append(index.reshape(1, 1))
    return decode(torch.cat(decoded, dim=1))

  return wrapper

## Greedy decoding

Greedy decoding is the simplest way to decode text from a trained model: at each step, take the most probable word.

In the equation:

$$P(\text{Text}|\text{Initial}) = \prod_{i=1}^{\text{Text length}} P(\text{Word}_i| \text{Word}_{j<i}, \text{Initial})$$

greedy decoding chooses $\text{Word}_i$ to maximize $P(\text{Word}_i| \text{Word}_{j<i}, \text{Initial})$. But it is often necessary to choose a less probable word to then reach very probable ones.

The following example comes from Hugging Face's article on text generation:

![Greedy decoding](https://huggingface.co/blog/assets/02_how-to-generate/greedy_search.png)

After "The", "nice" is chosen because it is the most probable word, while the complete text "The nice woman" has a lower probability (0.20) than the one we would have obtained by choosing "dog" ("The dog has", 0.36).

*Write the function `greedy(logits: torch.Tensor) -> torch.Tensor` that takes the model's logits and returns the index of the chosen word.*

*It uses [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html). The `decoding_loop` function expects a tensor of shape `()`.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def greedy(logits: torch.Tensor) -> torch.Tensor:
  return torch.argmax(logits, dim=-1)[0]


print(greedy("Je suis allé au" if lang == "fr" else "I went to the", 10))

## Testing on various inputs

*Write a function `test_decoding(function, *args, **kwargs) -> None` that takes a decoding method and its arguments, and prints the results of this method on the inputs below. Test the `greedy` method with it. What do you notice?*

In [ ]:
if lang == "fr":
  inputs = ["Je suis allé au",
            "Le train pour Lyon",
            "Comment vas-tu",
            "La recette de la tarte aux pommes commence par",
            "Ça va merci, tu devrais",
            "Le chat de la voisine",
            "En 2050, les voitures"]
else:
  inputs = ["I went to the",
            "The train to London",
            "How do you",
            "The apple pie recipe starts with",
            "I'm fine thank you, you should",
            "The neighbour's cat",
            "In 2050, cars"]

In [ ]:
# Your code here

### Solution

In [ ]:
def test_decoding(function, *args, **kwargs) -> None:
  generations = [function(prompt, *args, **kwargs)
                 for prompt in tqdm.auto.tqdm(inputs, desc="Prompts", leave=False)]
  for prompt, generation in zip(inputs, generations):
    print("—" * 80)
    print(f"Prompt: {prompt}")
    print(f"Generation: {generation}")
  print("—" * 80)


test_decoding(greedy, 50)

The answers "loop" quite quickly: they repeat the same sentence. This is a classic weakness of greedy decoding.

## Sampling

Greedy decoding results loop quickly and are often very generic. The following methods try to fix these flaws.

The first one is to **sample** from the model's probabilities, instead of always choosing the most probable word.

*Starting from the `greedy` function, instead of choosing the maximum element, compute a probability distribution with [`torch.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html), then use [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) to draw the next word from this distribution.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def sampling(logits: torch.Tensor) -> torch.Tensor:
  probabilities = torch.softmax(logits, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


test_decoding(sampling, 50)

## Temperature

Before the softmax, the logits can be divided by a **temperature** $T$:

- $T < 1$ sharpens the differences: the model more often picks the most probable words (more coherent, less varied);
- $T > 1$ flattens them: rare words are drawn more often (more varied, less coherent);
- $T \to 0$ amounts to greedy decoding.

*Add a `temperature` parameter to your sampling function in a new `temperature_sampling` function, and compare generations for $T = 0.5$ and $T = 1.5$.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def temperature_sampling(logits: torch.Tensor, temperature: float) -> torch.Tensor:
  probabilities = torch.softmax(logits / temperature, dim=-1)
  return torch.multinomial(probabilities, num_samples=1)[0, 0]


test_decoding(temperature_sampling, 50, temperature=0.5)
test_decoding(temperature_sampling, 50, temperature=1.5)

## Top-k sampling

A variation of sampling is to only draw among the `k` most probable words, to avoid generating a very improbable word.

*Starting from the `sampling` function, write the function `k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor` that implements this improvement with [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html). Test it with the `test_decoding` function.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def k_sampling(logits: torch.Tensor, k: int) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values, dim=-1)
  sampled = torch.multinomial(probabilities, num_samples=1)[0, 0]
  return indices[0, sampled]


test_decoding(k_sampling, 50, k=20)

## Top-p sampling

Another improvement is to only keep the most probable words whose cumulative probability reaches `p` (the *nucleus* of the distribution), and not the following ones. The number of candidate words then adapts to the model's confidence.

*Starting from the `k_sampling` function, add top-p sampling to it, in the function `k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor`. You can use [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html). Test it with the `test_decoding` function.*

In [ ]:
# Your code here

### Solution

In [ ]:
@decoding_loop
def k_p_sampling(logits: torch.Tensor, k: int, p: float) -> torch.Tensor:
  values, indices = torch.topk(logits, k=k, dim=-1)
  probabilities = torch.softmax(values[0], dim=-1)
  # Number of words needed for the cumulative probability to reach p
  n_kept = int((torch.cumsum(probabilities, dim=0) < p).sum()) + 1
  sampled = torch.multinomial(probabilities[:n_kept], num_samples=1)[0]
  return indices[0, sampled]


test_decoding(k_p_sampling, 100, k=20, p=0.85)

## Using the `transformers` functions

In practice, to generate text, use the `generate` method of the `transformers` library's models, which implements all these strategies (and others, such as beam search):

In [ ]:
def transformers_generate(prompt: str, length: int, temperature: float,
                          top_k: int, top_p: float) -> str:
  token_ids = encode(prompt)
  generated = model.generate(token_ids,
                             attention_mask=torch.ones_like(token_ids),
                             max_new_tokens=length,
                             do_sample=True,
                             temperature=temperature,
                             top_k=top_k,
                             top_p=top_p,
                             pad_token_id=tokenizer.eos_token_id)
  return decode(generated)


test_decoding(transformers_generate, length=100, temperature=1.0, top_k=30, top_p=0.95)